# Phase 3-1: YOLOv8 Feature Extraction

Extract intermediate features from YOLOv8 for adversarial detection analysis.

In [ ]:
import os
import sys
import pickle
import zipfile
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from tqdm import tqdm

# check if colab
IN_COLAB = 'google.colab' in sys.modules
print(f"Colab: {IN_COLAB}")
print(f"CUDA: {torch.cuda.is_available()}")

In [ ]:
if IN_COLAB:
    !pip install ultralytics -q
    from google.colab import drive
    drive.mount('/content/drive')

from ultralytics import YOLO

In [ ]:
# paths - change these for your setup
if IN_COLAB:
    DRIVE = "/content/drive/MyDrive/Colab Notebooks/data"
    STAGING = "/content/staged_data"
else:
    DRIVE = "/Users/tyreecruse/Desktop/CS230/Project/Data"
    STAGING = DRIVE

MODEL_PATH = f"{DRIVE}/training/results from training/weights/best.pt"
ZIPS_DIR = f"{DRIVE}/analysis/zips"
OUTPUT_DIR = f"{DRIVE}/analysis/yolo_features"

# layers to extract (9=SPPF backbone, 15/18/21=neck outputs)
LAYERS = [9, 15, 18, 21]

# datasets to process
DATASETS = [
    "clean",
    "fgsm_030", "fgsm_045", "fgsm_060", "fgsm_075", "fgsm_090", "fgsm_105",
    "gaussian_010", "gaussian_050", "gaussian_150", "gaussian_200", "gaussian_250",
    "patches",
]

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(STAGING, exist_ok=True)

print(f"Model: {MODEL_PATH}")
print(f"Output: {OUTPUT_DIR}")

In [ ]:
# load model
print("Loading model...")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = YOLO(MODEL_PATH)
model.model.to(device)
model.model.eval()
print(f"Loaded on {device}")

In [ ]:
# set up hooks to grab intermediate features
features = {}
hooks = []

def makeHook(layerIdx):
    def hook(module, inp, out):
        features[layerIdx] = out.detach()
    return hook

for idx in LAYERS:
    layer = model.model.model[idx]
    h = layer.register_forward_hook(makeHook(idx))
    hooks.append(h)

print(f"Registered {len(hooks)} hooks")

In [ ]:
def extractFeatures(imgPath):
    """Run image through model and grab pooled features from each layer."""
    global features
    features = {}  # reset
    
    # load and preprocess
    img = Image.open(imgPath).convert('RGB').resize((640, 640))
    imgTensor = torch.from_numpy(np.array(img)).float()
    imgTensor = imgTensor.permute(2, 0, 1) / 255.0
    imgTensor = imgTensor.unsqueeze(0).to(device)
    
    # forward pass
    with torch.no_grad():
        _ = model.model(imgTensor)
    
    # pool each layer to a vector
    pooled = {}
    for idx, feat in features.items():
        p = nn.functional.adaptive_avg_pool2d(feat, (1, 1))
        pooled[idx] = p.squeeze().cpu().numpy()
    
    # concatenate all layers
    allFeats = np.concatenate([pooled[i] for i in sorted(pooled.keys())])
    
    return allFeats, pooled

In [ ]:
def processDataset(name):
    """Unzip dataset, extract features, save to pickle."""
    print(f"\n{'='*50}")
    print(f"Processing: {name}")
    print('='*50)
    
    # unzip if needed
    zipPath = f"{ZIPS_DIR}/{name}.zip"
    stagePath = f"{STAGING}/{name}"
    
    if not os.path.exists(stagePath):
        print(f"Unzipping...")
        with zipfile.ZipFile(zipPath, 'r') as zf:
            zf.extractall(STAGING)
    
    # find images directory (handle different structures)
    imgDir = None
    for candidate in [f"{stagePath}/images", stagePath, f"{stagePath}/{name}/images"]:
        if os.path.exists(candidate):
            imgDir = candidate
            break
    
    if imgDir is None:
        print(f"Couldn't find images dir!")
        return None
    
    # get image files
    imgFiles = []
    for f in os.listdir(imgDir):
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            imgFiles.append(os.path.join(imgDir, f))
    imgFiles.sort()
    
    print(f"Found {len(imgFiles)} images")
    
    # extract features for all images
    allFeatures = []
    perLayer = {idx: [] for idx in LAYERS}
    
    for imgPath in tqdm(imgFiles):
        try:
            feat, layerFeats = extractFeatures(imgPath)
            allFeatures.append(feat)
            for idx in LAYERS:
                perLayer[idx].append(layerFeats[idx])
        except:
            pass  # skip bad images
    
    # stack into arrays
    featArray = np.vstack(allFeatures)
    perLayerArray = {idx: np.vstack(perLayer[idx]) for idx in LAYERS}
    
    # normalize (L2)
    norms = np.linalg.norm(featArray, axis=1, keepdims=True)
    featArray = featArray / (norms + 1e-8)
    
    for idx in perLayerArray:
        n = np.linalg.norm(perLayerArray[idx], axis=1, keepdims=True)
        perLayerArray[idx] = perLayerArray[idx] / (n + 1e-8)
    
    print(f"Feature shape: {featArray.shape}")
    
    # save
    data = {
        'features': featArray,
        'featuresPerLayer': perLayerArray,
        'nImages': len(imgFiles),
        'layers': LAYERS,
        'createdAt': datetime.now().isoformat(),
    }
    
    outPath = f"{OUTPUT_DIR}/{name}_yolo_features.pkl"
    if name == 'clean':
        outPath = f"{OUTPUT_DIR}/clean_yolo_features.pkl"
    
    with open(outPath, 'wb') as f:
        pickle.dump(data, f)
    
    print(f"Saved: {outPath}")
    return featArray.shape

In [ ]:
# process all datasets
results = {}

for name in DATASETS:
    try:
        shape = processDataset(name)
        results[name] = shape
    except Exception as e:
        print(f"Error: {e}")
        results[name] = None

In [ ]:
# remove hooks
for h in hooks:
    h.remove()
print("Hooks removed")

In [ ]:
print("\n" + "="*50)
print("SUMMARY")
print("="*50)

for name, shape in results.items():
    if shape:
        print(f"  {name}: {shape}")
    else:
        print(f"  {name}: FAILED")

# list output files
print(f"\nOutput files:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    if f.endswith('.pkl'):
        size = os.path.getsize(f"{OUTPUT_DIR}/{f}") / 1e6
        print(f"  {f}: {size:.1f} MB")

In [ ]:
# quick check
print("\nVerifying clean features...")
with open(f"{OUTPUT_DIR}/clean_yolo_features.pkl", 'rb') as f:
    data = pickle.load(f)

print(f"Shape: {data['features'].shape}")
print(f"Mean: {data['features'].mean():.4f}")
print(f"Std: {data['features'].std():.4f}")

norms = np.linalg.norm(data['features'], axis=1)
print(f"Norms: [{norms.min():.3f}, {norms.max():.3f}]")